There is a folder with name
```
C:\Users\kou12\University of Cambridge\Jose Lucas Taiyo Lee Rocha Santos - Cheeseboard\Raw Data\
```
Which have many folders with the name in the `YYYY-MM-DD` format
within each folder there is a subfolder called `movie`
And within that `movie` folder there are multiple files in the *.avi format.
Those *.avi file names are all in the format: `YYYY-MM-DD_{Animal_name}_trial_{n}.avi`

Below is a python code, that goes through all the sub-folder in the `Raw Data` folder,
And make a python dataframe, with columns:
* id (Animal name)
* date (in the YYYYMMDD format)

The script will also pick out two types of problems:
1. Unexpected `*.avi` filename: will raise any avi which voilate the filename convention `YYYY-MM-DD_{Animal_name}_trial_{n}.avi`
2. Non-sequential trial ID: Animal-dates with potentially missing trials, for some, maybe mistakes in the animal names:
Note: AF-20250608: Missing trial(s) in sequence: [1, 3, 4] means in AF-20250608 only trial 1,3,4 are present but 2 is missing. If there are repeating trials, like [1, 1, ... 7, 7, 8, 8] it is likely that the data is saved twice at different locations in the Raw Data folder

In [11]:
import os
import pandas as pd
import re

# Path to your Raw Data folder
base_path = r"D:\RawDataEdited"

# Regex to capture: YYYY-MM-DD_{Animal}_trial_{n}.avi
pattern = re.compile(r"(\d{4}-\d{2}-\d{2})_([^_]+)_trial_(\d+)\.avi")

records = []

# Walk through Raw Data folder
for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.endswith(".avi"):
            m = pattern.match(file)
            if m:
                date_str, animal_name, trial_num = m.groups()
                date = date_str.replace("-", "")  # YYYYMMDD
                # if date == '20250607' and animal_name == 'AD':
                #     print(f"Found: {file}")
                trial_num = int(trial_num)
                records.append({
                    "id": animal_name,
                    "date": date,
                    "trial": trial_num
                })
            else:
                print(f"⚠️ Filename format unexpected: {file}")

# Make DataFrame
df_trials = pd.DataFrame(records)

# Group by id/date to compute number of trials
def check_trials(trials, animal_name, date):
    trials_sorted = sorted(trials)
    n_trials = len(trials_sorted)
    max_trial = max(trials_sorted)
    if trials_sorted != list(range(1, max_trial + 1)):
        print(f"⚠️ {animal_name}-{date}: Missing trial(s) in sequence: {trials_sorted}")
    return n_trials

trial_info = (
    df_trials.groupby(["id", "date"])["trial"]
    .apply(list)
    .reset_index()
)

trial_info["n_trials"] = trial_info.apply(
    lambda row: check_trials(row["trial"], row["id"], row["date"]), axis=1
)

# Merge into final df (one row per id/date)
df = trial_info.drop(columns="trial").sort_values(by=["id", "date"]).reset_index(drop=True)

print(df.head())

# # Save CSV
# output_csv = os.path.join(base_path, "animal_data_with_trials.csv")
# df.to_csv(output_csv, index=False)
# print(f"✅ CSV saved to {output_csv}")


⚠️ AI-20250716: Missing trial(s) in sequence: [1, 2, 3, 4, 5, 6, 7, 8, 11]
⚠️ AJ-20250716: Missing trial(s) in sequence: [1, 2, 3, 4, 5, 6, 7, 8, 11]
⚠️ AL-20250716: Missing trial(s) in sequence: [1, 2, 3, 4, 5, 6, 7, 8, 11]
⚠️ AM-20250822: Missing trial(s) in sequence: [1, 2, 3, 4, 5, 6, 7, 8, 11]
⚠️ AN-20250822: Missing trial(s) in sequence: [1, 2, 3, 4, 5, 6, 7, 8, 11]
⚠️ AO-20250822: Missing trial(s) in sequence: [1, 2, 3, 4, 5, 6, 7, 8, 11]
  id      date  n_trials
0  A  20240930         6
1  A  20241001         5
2  A  20241205         6
3  A  20241206         6
4  A  20241207         8


Now, with the df created, look into another folder, named:
```
C:\Users\kou12\University of Cambridge\Jose Lucas Taiyo Lee Rocha Santos - Cheeseboard\ChR2
```
One level down in this folder, there are 6 sub folders, the names of these subfolders does not matter.
But now one more level into each of the 6 subfolders, there are more sub folders, named in the format:
`YYYY-MM-DD{type}{number}`, 
`type` means the session type, `number` is the days since the start of that session type
into these `YYYY-MM-DD{type}{number}` subfolders there are many *.avi files, again they are named
YYYY-MM-DD_{Animal_name}_trial_{n}.avi

Below is a script that go through each `*.avi` files, and
Find the matching animals - date in the `df`.
Add a column `type` in `df` to save the session type information,
And also add a column `day_in_type` to save the `number` information

If that animals - date pair does not exist in `df`, print :
f'{id} - {date} does not exist'

In [7]:
import re

base_chr2 = r"C:\Users\kou12\University of Cambridge\Jose Lucas Taiyo Lee Rocha Santos - Cheeseboard\ChR2"

# --- Now scan ChR2 folder ---
# Regex for session folder: YYYY-MM-DD{type}{number}
session_pattern = re.compile(r"(\d{4}-\d{2}-\d{2})([A-Za-z]+)(\d+)")

new_records = []

# Go two levels down in ChR2
for subfolder in os.listdir(base_chr2):
    subfolder_path = os.path.join(base_chr2, subfolder)
    if os.path.isdir(subfolder_path):
        for session_folder in os.listdir(subfolder_path):
            session_path = os.path.join(subfolder_path, session_folder)
            if os.path.isdir(session_path):
                m = session_pattern.match(session_folder)
                if not m:
                    continue
                date_str, session_type, session_num = m.groups()
                date = date_str.replace("-", "")
                day_in_type = int(session_num)

                # Look for AVI files inside
                for file in os.listdir(session_path):
                    if file.endswith(".avi"):
                        parts = file.split("_")
                        if len(parts) >= 3:
                            animal_name = parts[1]

                            # Check if (id, date) exists in df
                            mask = (df["id"] == animal_name) & (df["date"] == date)
                            if mask.any():
                                # If the type and day_in_type columns values is not nan, check if they match
                                if "type" in df.columns and "day_in_type" in df.columns:
                                    existing_type = df.loc[mask, "type"].values[0]
                                    existing_day = df.loc[mask, "day_in_type"].values[0]
                                    if pd.notna(existing_type) and pd.notna(existing_day):
                                        if existing_type != session_type or existing_day != day_in_type:
                                            print(f"Conflict for {animal_name} on {date}: existing ({existing_type}, {existing_day}) vs new ({session_type}, {day_in_type})")
                                            continue  # Skip this entry
                                else:
                                    df["type"] = pd.NA
                                    df["day_in_type"] = pd.NA
                                # Update df with type and day_in_type
                                df.loc[mask, "type"] = session_type
                                df.loc[mask, "day_in_type"] = day_in_type
                            else:
                                print(f"{animal_name} - {date} does not exist")

In [10]:
df

,id,date,n_trials
0,9,20240518,1
1,A,20240305,1
2,A,20240306,3
3,A,20240307,3
4,A,20240308,1
...,...,...,...
769,T,20241111,8
770,T,20241112,8
771,T,20241113,1
772,T,20241114,1


In [13]:
# Save df as csv
df.to_csv("animal_sessions_RawDataEdited.csv", index=False)

Below code to check if the positions.csv and track.png exist for each avi videos

In [10]:
import os
import re

# Base path
base_path = r"D:\RawDataEdited"

# Regex to capture: YYYY-MM-DD_{Animal}_trial_{n}.avi
avi_pattern = re.compile(r"(\d{4}-\d{2}-\d{2})_([^_]+)_trial_(\d+)\.avi")

# Walk through Raw Data folder
for root, dirs, files in os.walk(base_path):
    # Only check "movie" subfolders
    if os.path.basename(root) != "movie":
        continue

    # Path to tracking folder
    tracking_path = os.path.join(root, "tracking")
    if not os.path.exists(tracking_path):
        print(f"⚠️ Tracking folder missing in: {root}")
        continue

    # Collect all tracking files
    tracking_files = set(os.listdir(tracking_path))

    for file in files:
        if file.endswith(".avi"):
            m = avi_pattern.match(file)
            if not m:
                print(f"⚠️ Unexpected AVI filename: {file}")
                continue

            date_str, animal_name, trial_num = m.groups()
            date = date_str.replace("-", "")
            base_name = f"{date_str}_{animal_name}_trial_{trial_num}"

            # Expected files inside tracking/
            csv_file = f"{base_name}_positions.csv"
            png_file = f"{base_name}_trace.png"

            if csv_file not in tracking_files:
                print(f"{animal_name}-{date}: missing csv file ({csv_file})")
            if png_file not in tracking_files:
                print(f"{animal_name}-{date}: missing png file ({png_file})")


I-20241201: missing csv file (2024-12-01_I_trial_3_positions.csv)
I-20241201: missing png file (2024-12-01_I_trial_3_trace.png)
AF-20250624: missing csv file (2025-06-24_AF_trial_1_positions.csv)
AF-20250624: missing png file (2025-06-24_AF_trial_1_trace.png)
AJ-20250711: missing csv file (2025-07-11_AJ_trial_4_positions.csv)
AJ-20250711: missing png file (2025-07-11_AJ_trial_4_trace.png)
AN-20250725: missing png file (2025-07-25_AN_trial_6_trace.png)


In [3]:
file_set

{'2025-08-23_AM_trial_1.avi',
 '2025-08-23_AM_trial_2.avi',
 '2025-08-23_AM_trial_3.avi',
 '2025-08-23_AN_trial_1.avi',
 '2025-08-23_AO_trial_1.avi',
 '2025-08-23_AO_trial_2.avi',
 'convert.py'}